Embeddings das duas versões dos dados 

In [1]:
!pip install pandas
!pip install sentence-transformers
!pip install numpy

zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


In [2]:
import pandas as pd
df = pd.read_csv("data/com_texto_limpo.csv")

In [3]:
textos_limpos = df["claim"].tolist()
textos = df["Título da checagem"].tolist()

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings_originais = model.encode(textos, normalize_embeddings=True, show_progress_bar=True)
embeddings_limpos = model.encode(textos_limpos, normalize_embeddings=True, show_progress_bar=True)


/Users/aluno2/Residencia/RES-IA-Challenge-1/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12381.02it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Batches:   2%|▏         | 1/59 [00:00<00:32,  1.78it/s]

...

Batches:  71%|███████   | 42/59 [00:00<00:00, 45.19it/s]

Batches:  80%|███████▉  | 47/59 [00:01<00:00, 46.09it/s]

Batches:  90%|████████▉ | 53/59 [00:01<00:00, 47.71it/s]

Batches: 100%|██████████| 59/59 [00:01<00:00, 39.47it/s]

Batches: 100%|██████████| 59/59 [00:01<00:00, 41.48it/s]

In [5]:
import numpy as np

# 1. matrizes de similaridade
S_orig = embeddings_originais @ embeddings_originais.T
S_limpos = embeddings_limpos @ embeddings_limpos.T

# 2. Similaridade média (subtraímos 'n' para tirar a diagonal principal onde o valor é 1)
n = len(df)
media_orig = (S_orig.sum() - n) / (n**2 - n)
media_limpos = (S_limpos.sum() - n) / (n**2 - n)

print(f"Média de similaridade ORIGINAL: {media_orig:.4f}")
print(f"Média de similaridade LIMPA: {media_limpos:.4f}")

# 3. Análise Qualitativa de Vizinhos
idx = df[df["Título da checagem"].str.startswith("É falso que", na=False)].index[0]
print(f"\n📌 ALVO (Índice {idx}): {df['Título da checagem'].iloc[idx]}\n")

print("--- TOP 5 VIZINHOS (ORIGINAL) ---")
vizinhos_orig = np.argsort(-S_orig[idx])[1:6]
for v in vizinhos_orig:
    print(f"- {df['Título da checagem'].iloc[v]}")

print("\n--- TOP 5 VIZINHOS (LIMPO) ---")
vizinhos_limpos = np.argsort(-S_limpos[idx])[1:6]
for v in vizinhos_limpos:
    print(f"- {df['claim'].iloc[v]}")


np.save("data/emb_minilm.npy", embeddings_limpos)

Média de similaridade ORIGINAL: 0.2543
Média de similaridade LIMPA: 0.2131

📌 ALVO (Índice 7): É falso que Lula disse que o agronegócio deve ser "eliminado da Terra"

--- TOP 5 VIZINHOS (ORIGINAL) ---
- É falso que Lula disse que agronegócio deve ser ‘eliminado da Terra’
- É falso que Lula e MST disseram que o agronegócio deve ser eliminado da Terra
- Lula diz que agronegócio deve ser eliminado da Terra #boato
- É falso que Lula disse que vai acabar com o Pix
- É falso que Lula tenha prometido aumentar IR e confiscar poupanças

--- TOP 5 VIZINHOS (LIMPO) ---
- Lula disse que agronegócio deve ser ‘eliminado da Terra’
- Lula diz que agronegócio deve ser eliminado da Terra
- Lula e MST disseram que o agronegócio deve ser eliminado da Terra
- Lula diz que quer acabar com o MEI (regime de Microempreendedor Individual)
- Não há registro público de declaração de Lula nem do MST sobre eliminar agronegócio da Terra
